# Sales analysis
Source table: workspace.dbsf_2305122.sales_raw
Author: Ayush Raj
Purpose: revenue by region and by product

In [0]:
df = spark.table("workspace.dbsf_2305122.sales_raw")
display(df)

order_id,order_date,region,product,quantity,unit_price
1001,2026-01-05,North,Keyboard,3,45.0
1002,2026-01-05,South,Monitor,1,189.5
1003,2026-01-06,East,Keyboard,5,45.0
1004,2026-01-07,North,Mouse,10,17.25
1005,2026-01-08,West,Monitor,2,189.5
1006,2026-01-09,South,Docking Station,4,120.0
1007,2026-01-10,East,Mouse,7,17.25
1008,2026-01-11,North,Monitor,3,189.5
1009,2026-01-12,West,Keyboard,2,45.0
1010,2026-01-12,South,Mouse,15,17.25


This cell calculates total revenue and units sold for each product.

In [0]:
from pyspark.sql.functions import col, sum as _sum

by_product = (df
    .withColumn("revenue", col("quantity") * col("unit_price"))
    .groupBy("product")
    .agg(
        _sum("revenue").alias("revenue"),
        _sum("quantity").alias("units")
    )
    .orderBy(col("revenue").desc()))

display(by_product)

product,revenue,units
Monitor,1137.0,6
Docking Station,840.0,7
Mouse,552.0,32
Keyboard,450.0,10


In [0]:
dbutils.widgets.dropdown(
    name="region",
    defaultValue="North",
    choices=["North", "South", "East", "West"],
    label="Region"
)

This cell filters the sales data using the region selected in the widget.

In [0]:
chosen = dbutils.widgets.get("region")

print("Filtering on region:", chosen)

region_rows = (df
    .filter(col("region") == chosen)
    .withColumn("revenue", col("quantity") * col("unit_price")))

display(region_rows)

Filtering on region: South


order_id,order_date,region,product,quantity,unit_price,revenue
1002,2026-01-05,South,Monitor,1,189.5,189.5
1006,2026-01-09,South,Docking Station,4,120.0,480.0
1010,2026-01-12,South,Mouse,15,17.25,258.75


In [0]:
%run ./Lab03_Helpers

Helpers loaded by: 2305122@kiit.ac.in


In [0]:
priced = add_revenue(df)

display(priced.select(
    "order_id",
    "region",
    "product",
    "quantity",
    "unit_price",
    "revenue"
))

order_id,region,product,quantity,unit_price,revenue
1001,North,Keyboard,3,45.0,135.0
1002,South,Monitor,1,189.5,189.5
1003,East,Keyboard,5,45.0,225.0
1004,North,Mouse,10,17.25,172.5
1005,West,Monitor,2,189.5,379.0
1006,South,Docking Station,4,120.0,480.0
1007,East,Mouse,7,17.25,120.75
1008,North,Monitor,3,189.5,568.5
1009,West,Keyboard,2,45.0,90.0
1010,South,Mouse,15,17.25,258.75
